# 03 - Fine-tuning de BETO para ABSA

Fine-tuneamos `dccuchile/bert-base-spanish-wwm-uncased` usando el formato de
auxiliary sentence (Sun et al., 2019):

```
[CLS] reseña [SEP] aspecto [SEP]  -> {pos, neg, neu}
```

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from sklearn.model_selection import train_test_split

from src.aspects.extractor import extract_aspects, load_spacy_model
from src.classical.lexicon import load_lexicon, score_aspect_lexicon
from src.data.loader import load_reviews
from src.evaluation.metrics import evaluate_predictions
from src.transformers_models.beto import BETOAspectClassifier

## 1. Preparar dataset

In [ ]:
df = load_reviews("../data/sample/reviews_sample.csv")
nlp = load_spacy_model()
lexicon = load_lexicon()

def to_label(s):
    return 'pos' if s > 0.1 else 'neg' if s < -0.1 else 'neu'

texts, aspects, labels = [], [], []
for _, row in df.iterrows():
    for asp in extract_aspects(row['text'], nlp):
        texts.append(row['text'])
        aspects.append(asp)
        labels.append(to_label(score_aspect_lexicon(row['text'], asp, lexicon)))
print(f'Ejemplos: {len(texts)}')

## 2. Split entrenamiento / test

In [ ]:
t_tr, t_te, a_tr, a_te, y_tr, y_te = train_test_split(
    texts, aspects, labels, test_size=0.2, random_state=42, stratify=labels if min(labels.count(c) for c in set(labels)) >= 2 else None)

## 3. Cargar e instanciar BETO

**Nota:** la primera ejecución descarga ~440MB.

In [ ]:
clf = BETOAspectClassifier()
print('Device:', clf.device)

## 4. Entrenamiento

Configuración recomendada: `lr=2e-5`, `batch_size=16`, `epochs=3` (Sun et al., 2019).

In [ ]:
history = clf.fit(t_tr, a_tr, y_tr, epochs=3, batch_size=16, learning_rate=2e-5)
history

## 5. Evaluación

In [ ]:
preds = clf.predict(t_te, a_te)
evaluate_predictions(y_te, preds)

## 6. Guardar modelo

In [ ]:
clf.save('../models/beto')